In [1]:
!pip install -qq graphiti-core

In [2]:
import neo4j 

In [5]:
import asyncio
from neo4j import GraphDatabase 

NEO4J_URI = "neo4j://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "neo4j_embedding"

In [9]:
from openai import AsyncAzureOpenAI
from graphiti_core import Graphiti
from graphiti_core.llm_client import LLMConfig, OpenAIClient
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient

# Azure OpenAI configuration
from config import azure_config

# Create Azure OpenAI client for LLM
azure_llm_client = AsyncAzureOpenAI(
        api_key=azure_config.api_key,
        api_version=azure_config.api_version,
        azure_endpoint=azure_config.endpoint,
        azure_deployment=azure_config.model_name,
    )
     

# Embedding client for embeddings deployment
azure_embed_client = AsyncAzureOpenAI(
    api_key=azure_config.api_key,
    api_version=azure_config.api_version,
    azure_endpoint=azure_config.endpoint,
    azure_deployment=azure_config.embedding_model_name,
)

# Set up Graphiti
graphiti = Graphiti(
    NEO4J_URI,
    NEO4J_USERNAME,
    NEO4J_PASSWORD,
    llm_client=OpenAIClient(
        client=azure_llm_client,
    ),
    embedder=OpenAIEmbedder(
        config=OpenAIEmbedderConfig(
            embedding_model=azure_config.embedding_model_name,  # Your *embedding* model deployment!
        ),
        client=azure_embed_client,  # using the embedding deployment client
    ),
    cross_encoder=OpenAIRerankerClient(
        client=azure_llm_client,
    )
)

# Now you can use Graphiti with Azure OpenAI

In [10]:
import nest_asyncio
nest_asyncio.apply()

In [13]:
async def main():
    # Main function implementation will go here
    try:
        # Initialize the graph database with graphiti's indices. This only needs to be done once.
        await graphiti.build_indices_and_constraints()
        print('Indices and constraints built')
        # Additional code will go here
        
    finally:
        # Close the connection
        await graphiti.close()
        print('Connection closed')

 
if __name__ == '__main__':
    asyncio.run(main())

Indices and constraints built
Connection closed


In [ ]:
import json, os
from datetime import datetime, timezone
from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

# Episodes list containing both text and JSON episodes
episodes = [
    {
        'content': 'Kamala Harris is the Attorney General of California. She was previously '
        'the district attorney for San Francisco.',
        'type': EpisodeType.text,
        'description': 'podcast transcript',
        
    },
    {
        'content': 'As AG, Harris was in office from January 3, 2011 – January 3, 2017',
        'type': EpisodeType.text,
        'description': 'podcast transcript',
    },
    {
        'content': {
            'name': 'Gavin Newsom',
            'position': 'Governor',
            'state': 'California',
            'previous_role': 'Lieutenant Governor',
            'previous_location': 'San Francisco',
        },
        'type': EpisodeType.json,
        'description': 'podcast metadata',
    },
    {
        'content': {
            'name': 'Gavin Newsom',
            'position': 'Governor',
            'term_start': 'January 7, 2019',
            'term_end': 'Present',
        },
        'type': EpisodeType.json,
        'description': 'podcast metadata',
    },
]

# Add episodes to the graph
for i, episode in enumerate(episodes):
    await graphiti.add_episode(
        name=f'Freakonomics Radio {i}',
        episode_body=episode['content']
        if isinstance(episode['content'], str)
        else json.dumps(episode['content']),
        source=episode['type'],
        source_description=episode['description'],
        reference_time=datetime.now(timezone.utc),
    )
    print(f'Added episode: Freakonomics Radio {i} ({episode["type"].value})')


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: name_embedding)} {position: line: 4, column: 44, offset: 106} for query: '\n        MATCH (n:Entity)\n        WHERE n.group_id IS NOT NULL\n        WITH n, vector.similarity.cosine(n.name_embedding, $search_vector) AS score\n        WHERE score > $min_score\n        RETURN\n            n.uuid As uuid, \n            n.name AS name,\n            n.group_id AS group_id,\n            n.created_at AS created_at, \n            n.summary AS summary,\n            labels(n) AS labels,\n            properties(n) AS attributes\n            \n        ORDER BY score

Added episode: Freakonomics Radio 0 (text)
Added episode: Freakonomics Radio 1 (text)
Added episode: Freakonomics Radio 2 (json)
Added episode: Freakonomics Radio 3 (json)


In [54]:
# Perform a hybrid search combining semantic similarity and BM25 retrieval
print("\nSearching for: 'Who was the California Attorney General?'")
results = await graphiti.search('Who was the California Attorney General?')

# Print search results
print('\nSearch Results:')
for result in results:
    print(f'UUID: {result.uuid}')
    print(f'Fact: {result.fact}')
    if hasattr(result, 'valid_at') and result.valid_at:
        print(f'Valid from: {result.valid_at}')
    if hasattr(result, 'invalid_at') and result.invalid_at:
        print(f'Valid until: {result.invalid_at}')
    print('---')



Searching for: 'Who was the California Attorney General?'

Search Results:
UUID: 4826e65b-f093-4629-b14a-4d855d84cd55
Fact: Kamala Harris is the Attorney General of California.
Valid from: 2025-06-16 14:10:44.828999+00:00
---
UUID: f024ce92-d90a-4e8d-afbe-0da203aebedd
Fact: Harris was in office as Attorney General of California from January 3, 2011 to January 3, 2017
Valid from: 2011-01-03 00:00:00+00:00
Valid until: 2017-01-03 00:00:00+00:00
---
UUID: 21f07d8c-7766-4a8e-84c5-480e80c5fcee
Fact: Governor of California
Valid from: 2025-06-16 14:10:59.398528+00:00
---
UUID: 535ee6f0-8b6e-4cbf-be7c-44b0a1d73e30
Fact: Gavin Newsom was the Lieutenant Governor
---
UUID: c493fca3-c109-4d30-9f0e-2ac2237e54ee
Fact: Lieutenant Governor in San Francisco
---
UUID: 17c3e9cc-be88-4c0e-b70c-43141a3b73cc
Fact: She was previously the district attorney for San Francisco.
Valid until: 2025-06-16 14:10:44.828999+00:00
---
UUID: 645680cf-a1ae-4302-aee2-cb737faa03d8
Fact: Gavin Newsom is the Governor
Valid 

In [55]:
# Use the top search result's UUID as the center node for reranking
if results and len(results) > 0:
    # Get the source node UUID from the top result
    center_node_uuid = results[0].source_node_uuid

    print('\nReranking search results based on graph distance:')
    print(f'Using center node UUID: {center_node_uuid}')

    reranked_results = await graphiti.search(
        'Who was the California Attorney General?', center_node_uuid=center_node_uuid
    )

    # Print reranked search results
    print('\nReranked Search Results:')
    for result in reranked_results:
        print(f'UUID: {result.uuid}')
        print(f'Fact: {result.fact}')
        if hasattr(result, 'valid_at') and result.valid_at:
            print(f'Valid from: {result.valid_at}')
        if hasattr(result, 'invalid_at') and result.invalid_at:
            print(f'Valid until: {result.invalid_at}')
        print('---')
else:
    print('No results found in the initial search to use as center node.')



Reranking search results based on graph distance:
Using center node UUID: b3799141-bc74-41bd-adef-340ad6b1553a

Reranked Search Results:
UUID: 4826e65b-f093-4629-b14a-4d855d84cd55
Fact: Kamala Harris is the Attorney General of California.
Valid from: 2025-06-16 14:10:44.828999+00:00
---
UUID: f024ce92-d90a-4e8d-afbe-0da203aebedd
Fact: Harris was in office as Attorney General of California from January 3, 2011 to January 3, 2017
Valid from: 2011-01-03 00:00:00+00:00
Valid until: 2017-01-03 00:00:00+00:00
---
UUID: 17c3e9cc-be88-4c0e-b70c-43141a3b73cc
Fact: She was previously the district attorney for San Francisco.
Valid until: 2025-06-16 14:10:44.828999+00:00
---
UUID: 21f07d8c-7766-4a8e-84c5-480e80c5fcee
Fact: Governor of California
Valid from: 2025-06-16 14:10:59.398528+00:00
---
UUID: 535ee6f0-8b6e-4cbf-be7c-44b0a1d73e30
Fact: Gavin Newsom was the Lieutenant Governor
---
UUID: 645680cf-a1ae-4302-aee2-cb737faa03d8
Fact: Gavin Newsom is the Governor
Valid from: 2025-06-16 14:10:59.39

In [61]:
from graphiti_core.edges import EntityEdge

def edges_to_facts_string(entities: list[EntityEdge]):
    return '\n- ' + '\n- '.join([edge.fact for edge in entities])

edges_facts_string = edges_to_facts_string(reranked_results)
print(f'Edges Facts String:{edges_facts_string}')

Edges Facts String:
- Kamala Harris is the Attorney General of California.
- Harris was in office as Attorney General of California from January 3, 2011 to January 3, 2017
- She was previously the district attorney for San Francisco.
- Governor of California
- Gavin Newsom was the Lieutenant Governor
- Gavin Newsom is the Governor
- Lieutenant Governor in San Francisco


In [18]:
# Example: Perform a node search using _search method with standard recipes
print(
    '\nPerforming node search using _search method with standard recipe NODE_HYBRID_SEARCH_RRF:'
)

# Use a predefined search configuration recipe and modify its limit
node_search_config = NODE_HYBRID_SEARCH_RRF.model_copy(deep=True)
node_search_config.limit = 5  # Limit to 5 results

# Execute the node search
node_search_results = await graphiti._search(
    query='California Governor',
    config=node_search_config,
)

# Print node search results
print('\nNode Search Results:')
for node in node_search_results.nodes:
    print(f'Node UUID: {node.uuid}')
    print(f'Node Name: {node.name}')
    node_summary = node.summary[:100] + '...' if len(node.summary) > 100 else node.summary
    print(f'Content Summary: {node_summary}')
    print(f"Node Labels: {', '.join(node.labels)}")
    print(f'Created At: {node.created_at}')
    if hasattr(node, 'attributes') and node.attributes:
        print('Attributes:')
        for key, value in node.attributes.items():
            print(f'  {key}: {value}')
    print('---')



Performing node search using _search method with standard recipe NODE_HYBRID_SEARCH_RRF:

Node Search Results:
Node UUID: c823b763-df84-4bf5-9b5a-80d1a4281cb7
Node Name: Governor
Content Summary: Gavin Newsom is the Governor of California. Prior to his current role, he served as the Lieutenant G...
Node Labels: Entity
Created At: 2025-06-16 14:11:00.428946+00:00
Attributes:
  labels: ['Entity']
---
Node UUID: 47065a2b-4cdb-4c64-afb6-e53a9f0f4c46
Node Name: Gavin Newsom
Content Summary: Gavin Newsom is the Governor of California, having started his term on January 7, 2019. Before becom...
Node Labels: Entity
Created At: 2025-06-16 14:11:00.428897+00:00
Attributes:
  labels: ['Entity']
---
Node UUID: 5f9f6410-ad3c-446d-a05d-71e053f7ce52
Node Name: California
Content Summary: California is a state in the United States, currently governed by Gavin Newsom, who holds the positi...
Node Labels: Entity
Created At: 2025-06-16 14:11:00.428960+00:00
Attributes:
  labels: ['Entity']
---
Node UUID

In [29]:
from pydantic import BaseModel, Field

class Customer(BaseModel):
    """A customer of the service"""
    # name: str | None = Field(..., description="The name of the customer") # name cannot be used as an attribute for Customer as it is a protected attribute name.
    email: str | None = Field(..., description="The email address of the customer")
    email_domain: str | None = Field(..., description="The domain of the customer's email address")
    subscription_tier: str | None  = Field(..., description="The customer's subscription level")


class Product(BaseModel):
    """A product or service offering"""
    price: float | None  = Field(..., description="The price of the product or service")
    category: str | None  = Field(..., description="The category of the product")


In [30]:
entity_types = {"Customer": Customer, "Product": Product}
group_id = "customer_signups"
await graphiti.add_episode(
            name='Message',
            episode_body="New customer John (john@example.com) signed up for premium tier and purchased our Analytics Pro product ($199.99) from the Software category." ,
            reference_time=datetime.now(),
            source_description='Support Ticket Log',
            group_id=group_id,
            entity_types=entity_types,
        )


AddEpisodeResults(episode=EpisodicNode(uuid='a41a5737-98e6-4851-87fe-a909a31fc450', name='Message', group_id='customer_signups', labels=[], created_at=datetime.datetime(2025, 6, 16, 15, 0, 47, 529630, tzinfo=datetime.timezone.utc), source=<EpisodeType.message: 'message'>, source_description='Support Ticket Log', content='New customer John (john@example.com) signed up for premium tier and purchased our Analytics Pro product ($199.99) from the Software category.', valid_at=datetime.datetime(2025, 6, 16, 10, 0, 47, 529622), entity_edges=['f60182f2-a4a1-4e6e-8cab-796977422736']), nodes=[EntityNode(uuid='2534def8-e4bd-4c73-8d4c-539296470185', name='John', group_id='customer_signups', labels=['Entity', 'Customer'], created_at=datetime.datetime(2025, 6, 16, 14, 44, 11, 133679, tzinfo=<UTC>), name_embedding=[0.009958264417946339, 0.015256766229867935, -0.006417818367481232, 0.005070380866527557, -0.0029001848306506872, -0.02210649475455284, -0.0017717437585815787, 0.019369035959243774, 0.00359

In [51]:
edges_results = await graphiti.search('What products did John buy?, list them numerically',
                                      num_results=5,)
print('Edge Search Results:')
for edge in edges_results:
    print(f'Edge Type: {edge.name}')
    print(f'Edge fact: {edge.fact}')
    print(f'Edge Attributes: {edge.attributes.keys()}')


Edge Search Results:
Edge Type: PURCHASED
Edge fact: John purchased Analytics Pro
Edge Attributes: dict_keys(['fact_embedding'])


In [63]:
from graphiti_core.edges import EntityEdge

def edges_to_facts_string(entities: list[EntityEdge]):
    return '\n- ' + '\n- '.join([edge.fact for edge in entities])

edges_facts_string = edges_to_facts_string(edges_results)
print(f'Edges Facts String: {edges_facts_string}')

Edges Facts String: 
- John purchased Analytics Pro


In [49]:
node_search_config = NODE_HYBRID_SEARCH_RRF.model_copy(deep=True)
node_search_config.limit = 10  # Limit to 5 results
# Execute the node search
node_search_results = await graphiti._search(
    query='What products did John buy?, list them numerically',
    config=node_search_config,
)
# Print node search results
print('\nNode Search Results:')
for node in node_search_results.nodes:
    print(f'Node UUID: {node.uuid}')
    print(f'Node Name: {node.name}')
    node_summary = node.summary[:100] + '...' if len(node.summary) > 100 else node.summary
    print(f'Content Summary: {node_summary}')
    print(f"Node Labels: {', '.join(node.labels)}")
    print(f'Created At: {node.created_at}')
    if hasattr(node, 'attributes') and node.attributes:
        print('Attributes:')
        for key, value in node.attributes.items():
            print(f'  {key}: {value}')
    print('---')



Node Search Results:
Node UUID: 2534def8-e4bd-4c73-8d4c-539296470185
Node Name: John
Content Summary: John is a new customer who signed up for the premium tier and purchased the Analytics Pro product fo...
Node Labels: Entity, Customer
Created At: 2025-06-16 14:44:11.133679+00:00
Attributes:
  subscription_tier: premium
  email: john@example.com
  labels: ['Entity', 'Customer']
  name_: John
  email_domain: example.com
---
Node UUID: a8e6b709-3b7e-4a21-bc1f-b548c865fec9
Node Name: Analytics Pro
Content Summary: Analytics Pro is a premium software product purchased by new customer John (john@example.com). It is...
Node Labels: Entity, Product
Created At: 2025-06-16 14:44:11.133800+00:00
Attributes:
  category: Software
  price: 199.99
  labels: ['Entity', 'Product']
---
